# Data Mining - Project 

##### Work done by
- Roberto Rolão de Castro 107133
- Tiago Caridade Gomes 108307

## Tracking the Evolution of E-commerce Websites in Portugal 
**Description**: Study how e-commerce websites in Portugal, such as those related to retail (e.g., Worten, Fnac) have changed in terms of offerings and products availability.  
**Data** : Gather versions of popular e-commerce sites over time from Arquivo.pt.

In [20]:
import requests
import pandas as pd

def read_list_from_file(filename):
    """Reads a list of items from a file, one item per line."""
    with open(filename, 'r') as file:
        return [line.strip() for line in file if line.strip()]

def fetch_all_results(websites, keywords):
    url = "https://arquivo.pt/textsearch"
    all_data = []
    seen_urls = set()  # To track the URLs that have already been added

    for website in websites:
        for keyword in keywords:
            print(f"Starting search for '{keyword}' on {website}")
            base_params = {
                "from": "20200101000000",
                "to": "20240101000000",
                "maxItems": 500,
                "q": keyword,
                "siteSearch": website,
                "dedupValue": 0,
                "dedupField": "url",
                "offset": 0
            }

            # Fetch data for each combination of website and keyword
            while True:
                response = requests.get(url, params=base_params)
                if response.status_code != 200:
                    print(f"Failed to fetch data at offset {base_params['offset']}. Status code: {response.status_code}")
                    break

                data = response.json()
                if 'response_items' in data and data['response_items']:
                    current_data = pd.DataFrame(data['response_items'])

                    # Filter out duplicates based on originalURL
                    new_data = current_data[~current_data['linkToExtractedText'].isin(seen_urls)]
                    if not new_data.empty:
                        all_data.append(new_data)
                        # Add the new originalURLs to the seen set
                        seen_urls.update(new_data['linkToExtractedText'])
                        print(f"Added {len(new_data)} new items for '{keyword}' on {website} at offset {base_params['offset']}")
                    else:
                        print(f"No new items found for '{keyword}' on {website} at offset {base_params['offset']}")

                    # Check if fewer results were returned than maxItems, indicating the last page
                    if len(current_data) < base_params['maxItems']:
                        print("Reached the last page of data.")
                        break

                    base_params['offset'] += base_params['maxItems']
                else:
                    print("No more data available or reached API limit.")
                    break

            # Save the results for the current website and keyword
            if all_data:
                final_data = pd.concat(all_data, ignore_index=True)
                # Drop duplicates from the final concatenated DataFrame based on originalURL
                final_data = final_data.drop_duplicates(subset=['linkToExtractedText'])
                output_filename = f"./Results/{website.replace('https://', '').replace('.', '_')}_{keyword}.csv"
                final_data.to_csv(output_filename, index=False)
                print(f"Data saved to {output_filename}")
                all_data.clear()  # Clear the list for the next combination
            else:
                print(f"No data found for '{keyword}' on {website}")

def main():
    # Read websites and keywords from files
    websites = read_list_from_file('websites.txt')
    keywords = read_list_from_file('categories.txt')

    # Fetch all results
    fetch_all_results(websites, keywords)

if __name__ == "__main__":
    main()


Starting search for 'Telemóvel' on https://www.worten.pt
Added 500 new items for 'Telemóvel' on https://www.worten.pt at offset 0
Added 500 new items for 'Telemóvel' on https://www.worten.pt at offset 500
Added 500 new items for 'Telemóvel' on https://www.worten.pt at offset 1000
Added 500 new items for 'Telemóvel' on https://www.worten.pt at offset 1500
No more data available or reached API limit.
Data saved to ./Results/www_worten_pt_Telemóvel.csv
Starting search for 'Smartphone' on https://www.worten.pt
Added 496 new items for 'Smartphone' on https://www.worten.pt at offset 0
Added 481 new items for 'Smartphone' on https://www.worten.pt at offset 500
Added 472 new items for 'Smartphone' on https://www.worten.pt at offset 1000
Added 489 new items for 'Smartphone' on https://www.worten.pt at offset 1500
No more data available or reached API limit.
Data saved to ./Results/www_worten_pt_Smartphone.csv
Starting search for 'Portátil' on https://www.worten.pt
Added 376 new items for 'Portá

WEB Scrapping

In [18]:
import os
from bs4 import BeautifulSoup

def scrape_saved_files(results_dir):
    """Performs web scraping on the saved CSV files to extract product title and price."""
    
    scraped_data = []
    if not os.path.exists('./Scraped_Results'):
        os.makedirs('./Scraped_Results')

    for file_name in os.listdir(results_dir):
        if file_name.endswith('.csv'):
            file_path = os.path.join(results_dir, file_name)
            print(f"Scraping data from file: {file_path}")
            df = pd.read_csv(file_path)

            for _, row in df.iterrows():
                archive_url = row.get("linkToNoFrame")
                if archive_url:
                    try:
                        print(f"Scraping data from: {archive_url}")
                        page = requests.get(archive_url)
                        soup = BeautifulSoup(page.content, 'html.parser')

                        product_title = "N/A"
                        price = "N/A"
                        
                        site = row.get("siteSearch", '').lower()
                        
                        if 'fnac' in site:
                            product_title = soup.find(class_='f-productHeader-Title').get_text(strip=True) if soup.find(class_='f-productHeader-Title') else "N/A"
                            price = soup.find(class_='f-priceBox-price').get_text(strip=True) if soup.find(class_='f-priceBox-price') else "N/A"
                        elif 'worten' in site:
                            product_title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "N/A"
                            price = soup.find(class_='w-product-price__main').get_text(strip=True) if soup.find(class_='w-product-price__main') else "N/A"
                        elif 'radiopopular' in site:
                            product_title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "N/A"
                            price = soup.find(class_='current-price').get_text(strip=True) if soup.find(class_='current-price') else "N/A"
                        elif 'mediamarkt' in site:
                            product_title = soup.find('h1', class_='title').get_text(strip=True) if soup.find('h1', class_='title') else "N/A"
                            price = soup.find(class_='price').get_text(strip=True) if soup.find(class_='price') else "N/A"
                        elif 'kuantokusta' in site:
                            product_title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "N/A"
                            price = soup.find(class_='product-price').get_text(strip=True) if soup.find(class_='product-price') else "N/A"

                        scraped_data.append({
                            "title": product_title,
                            "price": price,
                            "category": row.get("q"),
                            "timestamp": row.get("timestamp"),
                            "website": row.get("siteSearch"),
                            "originalURL": row.get("originalURL")
                        })
                    except Exception as e:
                        print(f"Error scraping {archive_url}: {e}")
                        continue

            # Save scraped data to a new CSV
            if scraped_data:
                scraped_df = pd.DataFrame(scraped_data)
                scraped_output_file = f"./Scraped_Results/scraped_{file_name}"
                scraped_df.to_csv(scraped_output_file, index=False)
                print(f"Scraped data saved to {scraped_output_file}")
                scraped_data.clear()


def main():
    scrape_saved_files('./Results')

if __name__ == "__main__":
    main()

Scraping data from file: ./Results/www_fnac_pt_Smartphone.csv
Scraping data from: https://arquivo.pt/noFrame/replay/20180719181639/https://www.fnac.pt/Smartphone-Land-Rover-Explore-Telemovel-SmartPhone-Android/a3978851
Scraping data from: https://arquivo.pt/noFrame/replay/20200511111902/https://www.fnac.pt/Smartphone-Huawei-P8-Lite-Black-Telemovel-SmartPhone-Android/a874919
Scraped data saved to ./Scraped_Results/scraped_www_fnac_pt_Smartphone.csv
Scraping data from file: ./Results/www_kuantokusta_pt_Smartphone.csv
Scraping data from: https://arquivo.pt/noFrame/replay/20190324042505/https://www.kuantokusta.pt/search?q=smartphone
Scraping data from: https://arquivo.pt/noFrame/replay/20190423134158/https://www.kuantokusta.pt/search?q=smartphone&pag=1
Scraping data from: https://arquivo.pt/noFrame/replay/20190423134212/https://www.kuantokusta.pt/search?q=smartphone&pag=3
Scraping data from: https://arquivo.pt/noFrame/replay/20190423134219/https://www.kuantokusta.pt/search?q=smartphone&pag

KeyboardInterrupt: 